In [2]:
using GeoStats
using DataFrames
using CSV
using Proj4 
using Makie
using Plots
using Parquet

In [3]:
p13u_meta = DataFrame(read_parquet("../data/p13u_smbl_noev_metadata.parquet"))
select!(p13u_meta, Not([:substation,:has_battery,:has_pv]))

,bus,feeder,lat,lon
,String?,String?,Float64?,Float64?
1,p13udm19529,p13udt4524-p13uhs22_1247x,37.7877,-122.226
2,p13udm19530,p13udt4524-p13uhs22_1247x,37.7873,-122.226
3,p13udm19531,p13udt4524-p13uhs22_1247x,37.7879,-122.227
4,p13udm19532,p13udt4524-p13uhs22_1247x,37.7876,-122.226
5,p13udm19533,p13udt4524-p13uhs22_1247x,37.7875,-122.226
6,p13udm19534,p13udt4524-p13uhs22_1247x,37.7875,-122.226
7,p13udm19535,p13udt4524-p13uhs22_1247x,37.7873,-122.226
8,p13udm19536,p13udt4524-p13uhs22_1247x,37.7874,-122.226
9,p13udm19584,p13udt4524-p13uhs22_1247x,37.7878,-122.225


In [27]:
lines = CSV.read("../data/p13u_topology.csv", DataFrame; header=0)
skp = Vector{Float64}(undef,size(lines)[1]).=NaN
rename!(lines, :Column1 => :Bus1)
rename!(lines, :Column2 => :Bus2)
rename!(lines, :Column3 => :Length)
lines.Bus1 = lstrip.(rstrip.(lines.Bus1))
lines.Bus2 = lstrip.(rstrip.(lines.Bus2))

24611-element Vector{SubString{String15}}:
 "p13ulv82093"
 "p13ulv83140"
 "p13ulv3855"
 "p13ulv3856"
 "p13ulv22175"
 "p13udm6772"
 "p13udm6770"
 "p13udt1521lv"
 "p13ulv82103"
 "p13ulv22160"
 "p13ulv83145"
 "p13ulv83163"
 "p13udm6777"
 ⋮
 "p13udt29954lv"
 "p13udt29956lv"
 "p13udt29962lv"
 "p13udt30128lv"
 "p13udt30130lv"
 "p13udt30131lv"
 "p13udt30132lv"
 "p13udt30134lv"
 "p13udt30135lv"
 "p13udt30136lv"
 "p13udt30137lv"
 "p13udt30140lv"

In [28]:
bus_xy = Proj4.transform(Proj4.Projection("+proj=longlat +ellps=WGS84 +datum=WGS84 +no_defs"), Proj4.Projection("+proj=merc +a=6378137 +b=6378137 +lat_ts=0.0 +lon_0=0.0 +x_0=0.0 +y_0=0 +k=1.0 +units=m +nadgrids=@null +wktext  +no_defs"), hcat(p13u_meta.lon, p13u_meta.lat))
p13u_meta.x = bus_xy[:,1]
p13u_meta.y = bus_xy[:,2]

24579-element Vector{Float64}:
 4.549473065940808e6
 4.549428611833943e6
 4.549499587944485e6
 4.549458944083403e6
 4.549444822211358e6
 4.549451253661732e6
 4.549416588934573e6
 4.549436744038895e6
 4.54948933530915e6
 4.549479242579594e6
 4.549469139331276e6
 4.549566038775395e6
 4.54954862227411e6
 ⋮
 4.549521123035624e6
 4.549752942757781e6
 4.54987963160381e6
 4.549879994816717e6
 4.549792427589786e6
 4.549780942390828e6
 4.549922924975933e6
 4.550124450247466e6
 4.550362046914155e6
 4.550571127225045e6
 4.5506791569353705e6
 4.550660393792506e6

In [29]:
lines = innerjoin(lines, select(p13u_meta, [:bus,:x,:y]), on=:Bus1=>:bus)
rename!(lines, :x => :x1)
rename!(lines, :y => :y1)
lines = innerjoin(lines, select(p13u_meta, [:bus,:x,:y]), on=:Bus2=>:bus)
rename!(lines, :x => :x2)
rename!(lines, :y => :y2)

,Bus1,Bus2,Length,x1,y1,x2,y2
,SubStrin…,SubStrin…,Float64,Float64,Float64,Float64,Float64
1,p13udt1521lv,p13ulv82093,0.012353,-1.36046e7,4.55105e6,-1.36046e7,4.55106e6
2,p13udt1521lv,p13ulv83140,0.0129155,-1.36046e7,4.55105e6,-1.36046e7,4.55106e6
3,p13udm6769,p13ulv3855,0.0121367,-1.36046e7,4.55107e6,-1.36046e7,4.55108e6
4,p13udm6770,p13ulv3856,0.0107118,-1.36046e7,4.55108e6,-1.36046e7,4.55109e6
5,p13udm6772,p13ulv22175,0.00830569,-1.36045e7,4.55111e6,-1.36045e7,4.55111e6
6,p13udm6770,p13udm6772,0.0291177,-1.36046e7,4.55108e6,-1.36045e7,4.55111e6
7,p13udm6769,p13udm6770,0.00979012,-1.36046e7,4.55107e6,-1.36046e7,4.55108e6
8,p13udm6769,p13udt1521lv,0.00979012,-1.36046e7,4.55107e6,-1.36046e7,4.55105e6
9,p13udm6774,p13ulv82103,0.0133346,-1.36046e7,4.55105e6,-1.36046e7,4.55104e6


### Generate Contour Plots

In [11]:
function circle(x,y,r)
    t = LinRange(0, 2*pi, 500)
    x .+ r*sin.(t), y .+ r*cos.(t)
end

circle (generic function with 1 method)

In [64]:
function voltage_contour(filename)
    p13u = CSV.read("../data/" * filename * ".csv", DataFrame)
    p13u = innerjoin(select(p13u, [:bus, :voltage]), select(p13u_meta, [:bus,:feeder, :x,:y]), on="bus")
    p13u = filter(:voltage => v -> (v > 0.0), p13u)

    Nx=574; 
    Ny=775;
    margin = .0001
    G=CartesianGrid((minimum(p13u[!,:x])-margin,minimum(p13u[!,:y])-margin),
                    (maximum(p13u[!,:x])+margin,maximum(p13u[!,:y])+margin),dims=(Nx,Ny));
    S=georef(p13u,(:x,:y));
    problem=EstimationProblem(S,G,:voltage);

    # Shepard/Overbye heatmap algorithm is IDW 
    solver=IDW(:voltage=>(power=2,distance=Euclidean(),neighbors=246))
    solution = solve(problem, solver)

    x = repeat([G.origin.coords[1]:G.spacing[1]:G.origin.coords[1]+G.spacing[1]*(G.dims[1]-1);],outer=100)
    y = repeat([G.origin.coords[2]:G.spacing[2]:G.origin.coords[2]+G.spacing[2]*(G.dims[2]-1);],inner=100)
    voltage_grad=cgrad([RGB(0,20/255,94/255),:blue,:white,:red,RGB(114/255,28/255,16/255)], [0.05,0.05, 0.5, 0.95, 0.95] )
    
    # Plot the heatmap
    Plots.heatmap(G, solution[:voltage],c=voltage_grad, clim=(0.9455,1.0545), border=:none, size=(800,1080),ytickfontsize=10)
    Plots.plot!(collect(Iterators.flatten(zip(lines.x1,lines.x2,skp))),collect(Iterators.flatten(zip(lines.y1,lines.y2,skp))),
    color="black", linealpha=0.05, legend=false)
    png(filename * ".png")
    
    # Plot the key, outlining the violations
    violations = filter(:voltage => v -> (v > 1.05) || (v < 0.95), p13u)
    Plots.plot!(circle.(violations.x, violations.y, 125.0), linecolor="black", fillalpha=0, lw=0.5, linealpha=0.5)
    png(filename * "_key.png")
    
    # Plot the histograms
    histogram(p13u.voltage, bins=20, xlim=[minimum(p13u.voltage)-0.005,maximum(p13u.voltage)+0.005], legend=false, normed=true, yaxis=:log)
    png(filename * "_voltage_histogram.png")
    
    histogram(solution[:voltage], bins=20, xlim=[minimum(p13u.voltage)-0.005,maximum(p13u.voltage)+0.005],legend=false,normed=true,yaxis=:log)
    png(filename * "_heatmap_histogram.png")
end

voltage_contour (generic function with 1 method)

In [65]:
voltage_contour("Contour_1")
voltage_contour("Contour_2")
voltage_contour("Contour_3")

In [78]:
function outliers(v)
    return (1.2+abs(v-1)*25)^2.25 
end

function alpha(v)
    if (abs(v-1)<0.05) 
        return 0.65
    end
    return 1.0
end

function voltage_glyphs(filename)
    p13u = CSV.read("../data/" * filename * ".csv", DataFrame)
    p13u = innerjoin(select(p13u, [:bus, :voltage]), select(p13u_meta, [:bus,:feeder, :x,:y]), on="bus")
    p13u = filter(:voltage => v -> (v > 0.0), p13u)
    voltage_grad=cgrad([RGB(0,20/255,94/255),:blue,:white,:red,RGB(114/255,28/255,16/255)], [0.05,0.05, 0.5, 0.95, 0.95] )

    Nx=574; 
    Ny=775;
    margin = .0001
    G=CartesianGrid((minimum(p13u[!,:x])-margin,minimum(p13u[!,:y])-margin),
                    (maximum(p13u[!,:x])+margin,maximum(p13u[!,:y])+margin),dims=(Nx,Ny));

    p13u.order = abs.(p13u.voltage.-1)
    d=sort(p13u, [:order])
    Plots.heatmap(G, ones(Nx*Ny),c=voltage_grad, clim=(0.945,1.055), border=:none, size=(800,1000))
    Plots.plot!(collect(Iterators.flatten(zip(lines.x1,lines.x2,skp))),collect(Iterators.flatten(zip(lines.y1,lines.y2,skp))), color="black", linealpha=0.05, legend=false, border=:none)
    Plots.scatter!(d.x, d.y, markersize=outliers.(d.voltage),markerstrokewidth=0.02,markeralpha=alpha.(d.voltage), marker_z=d.voltage,c=voltage_grad, clim=(0.945, 1.055))
    png(filename * ".png")
end

voltage_glyphs (generic function with 1 method)

In [79]:
voltage_glyphs("Glyph_1")
voltage_glyphs("Glyph_2")
voltage_glyphs("Glyph_3")